# Training from scratch, using data from sampling

combined - data from sampling with forced higher accepcance, data from standard posterior sampling, data from sampling with high noise (i.e. higher weight of prior)

In [ ]:
# training NN on all collected data, using MINIBATCHES
from pathlib import Path
import numpy as np

from surrDAMH.surrogates.torch_perceptron_minibatches import PyTorchNNOngoingUpdater2 as PyTorchNNOngoingUpdater
from wrapper_grf import observations as ref_obs

# Base directory containing the saved surrogate state from the simple 2-parameter test.
base_dir = Path('nn_data')

# Input state to load first.
checkpoint_path = base_dir / 'surrogate_checkpoint.pt'
training_data_path = base_dir / 'surrogate_training_data.npz'
training_data_path_2 = base_dir / 'surrogate_training_data2.npz'  # additional data, will be joined together
test_data_path = base_dir / 'surrogate_test_data.npz'

# Output prefix for newly saved states.
output_prefix = 'from_scratch_tanh_adamw_128'

SURROGATE_OUTPUT_MEAN = np.asarray(ref_obs, dtype=np.float32).reshape(72)
SURROGATE_OUTPUT_SCALE = np.full((72,), 40.0, dtype=np.float32)

# The architecture must match the saved checkpoint.
updater_kwargs = dict(
    no_parameters=45,
    no_observations=72,
    hidden_layer_sizes=(128, 128, 128),
    solver="adamw",
    activation="tanh",
    learning_rate=3e-4,
    iterations_batch=100,
    loss_target=1e-6,
    device="cpu",
    verbose=False,
    # seed=42,
    output_mean=SURROGATE_OUTPUT_MEAN,
    output_scale=SURROGATE_OUTPUT_SCALE,
    batch_size=1024,
    replay_ratio=1.0,
    replay_max_old_samples=4096,
    train_on_added_data=False,
    shuffle_batches=True,
    gradient_clip_norm=10.0,
    weight_decay=1e-4,
)

print('Checkpoint exists:', checkpoint_path.exists())
print('Training data exists:', training_data_path.exists())
print('Test data exists:', test_data_path.exists())

with np.load(training_data_path) as loaded:
    print(loaded.files)
    parameters = loaded['parameters']
    observations = loaded['observations']
    weights = loaded['weights'] if 'weights' in loaded else np.ones((parameters.shape[0], 1), dtype=np.float32)

print('parameters shape:', parameters.shape)
print('observations shape:', observations.shape)
print('weights shape:', weights.shape)

with np.load(training_data_path_2) as loaded:
    print(loaded.files)
    parameters_2 = loaded['parameters']
    observations_2 = loaded['observations']
    weights_2 = loaded['weights'] if 'weights' in loaded else np.ones((parameters_2.shape[0], 1), dtype=np.float32)

# join the two datasets together
parameters = np.concatenate([parameters, parameters_2], axis=0)
observations = np.concatenate([observations, observations_2], axis=0)
weights = np.concatenate([weights, weights_2], axis=0)
print('parameters shape after joining:', parameters.shape)
print('observations shape after joining:', observations.shape)
print('weights shape after joining:', weights.shape)

with np.load(test_data_path) as loaded:
    print(loaded.files)
    parameters_test = loaded['test_parameters']
    observations_test = loaded['test_observations']
    weights_test = loaded['test_weights'] if 'test_weights' in loaded else np.ones((parameters_test.shape[0], 1), dtype=np.float32)

print('parameters test shape:', parameters_test.shape)
print('observations test shape:', observations_test.shape)
print('weights test shape:', weights_test.shape)

def build_updater():
    return PyTorchNNOngoingUpdater(**updater_kwargs)

updater = build_updater()
# loaded_arrays = updater.load_state(str(checkpoint_path), str(training_data_path), load_optimizer=True) # both data and checkpoint
loaded_arrays = updater.load_training_data(str(training_data_path)) # only data, train from scratch with loaded data and architecture, no optimizer state

print('Loaded snapshot count:', loaded_arrays[0].shape[0] if loaded_arrays is not None else 0)
initial_loss = updater.get_loss_on_data(parameters, observations)
print('Initial loss on loaded training data:', initial_loss)

Checkpoint exists: False
Training data exists: True
Test data exists: True
['parameters', 'observations', 'weights', 'no_parameters', 'no_observations', 'num_snapshots']
parameters shape: (59142, 45)
observations shape: (59142, 72)
weights shape: (59142, 1)
['parameters', 'observations', 'weights', 'no_parameters', 'no_observations', 'num_snapshots']
parameters shape after joining: (60871, 45)
observations shape after joining: (60871, 72)
weights shape after joining: (60871, 1)
['test_parameters', 'test_observations', 'test_log_posterior', 'test_weights']
parameters test shape: (256, 45)
observations test shape: (256, 72)
weights test shape: (256, 1)


NameError: name 'PyTorchNNOngoingUpdater' is not defined

In [ ]:
# define function to calculate gradient on test data (to test gradient stability during training)
def calculate_test_derivatives(updater, parameters_test):
    evaluator = updater.get_evaluator()
    res = []
    for i in range(15):
        par = parameters_test[i,:]
        jacobian = evaluator.jacobian(par)
        res += [jacobian[0][15,-4]]
    return np.array(res)

import surrDAMH
likelihood = surrDAMH.distributions.Normal(mean=ref_obs, sd=40.0)

def grad_in_zero(updater):
    parameters = np.zeros((45,), dtype=np.float32)
    evaluator = updater.get_evaluator()
    jacobian, evaluation = evaluator.jacobian(parameters)
    res = - jacobian.T @ likelihood.grad_logpdf(evaluation)
    return res[-15:]

In [ ]:
def plot_history(history):
    # extract loss from history and create a numpy array of loss values:
    loss_values = np.array([entry['loss'] for entry in history])
    test_loss_values = np.array([entry['test_loss'] for entry in history])
    print('Loss values:', loss_values)
    print('Test loss values:', test_loss_values)

    # plot loss values:
    import matplotlib.pyplot as plt
    # two subplots in one figure, first for loss values, second for test loss values:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 16))
    ax1.plot(loss_values, label='Training Loss')
    ax1.set_yscale('log')
    ax1.set_xlabel('Training Iteration')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss Over Iterations')
    # fine grid:
    ax1.grid(which='both', linestyle='--', linewidth=0.5)
    ax1.legend()
    ax2.plot(test_loss_values, label='Test Loss', color='orange')
    ax2.set_yscale('log')
    ax2.set_xlabel('Training Iteration')
    ax2.set_ylabel('Loss')
    ax2.set_title('Test Loss Over Iterations')
    # fine grid:
    ax2.grid(which='both', linestyle='--', linewidth=0.5)
    ax2.legend()
    plt.tight_layout()
    plt.savefig(base_dir / f'{output_prefix}_loss_plot.png')
    plt.show()


In [ ]:
history = []
current_checkpoint = checkpoint_path
current_training_data = training_data_path

for step in range(1, 3):
    # Continue training in memory.
    for i in range(100):
        for _ in range(10):
            updater.train()
        current_loss = updater.get_loss_on_data(parameters, observations)
        test_loss = updater.get_loss_on_data(parameters_test, observations_test)
        print(f'Step {step} {i}: loss={current_loss:.8e}, test_loss={test_loss:.8e}')
        res = calculate_test_derivatives(updater, parameters_test)
        # print all 15 values in 1 line with 8 decimal places:
        print('Test derivatives:', ' '.join([f'{x:.3e}' for x in res]))
        res = grad_in_zero(updater)
        print('Grad in zero:', ' '.join([f'{x:.3e}' for x in res]))

        history.append({
            'step': step,
            'loss': current_loss,
            'test_loss': test_loss,
        })
    plot_history(history)
    
    current_loss = updater.get_loss_on_data(parameters, observations)
    new_checkpoint = base_dir / f'{output_prefix}_{step}.pt'
    new_training_data = base_dir / f'{output_prefix}_{step}.npz'

    updater.save_state(str(new_checkpoint), str(new_training_data))


    current_checkpoint = new_checkpoint
    current_training_data = new_training_data

    print(f'Step {step}: loss={current_loss:.8e}')
    print(f'  saved checkpoint: {new_checkpoint}')
    print(f'  saved training data: {new_training_data}')

Step 1 0: loss=1.37316895e+03, test_loss=6.36937031e+04
Test derivatives: -1.471e+00 -1.185e+00 -3.555e-01 -2.645e+00 -9.677e-01 -1.824e+00 -1.027e+00 -2.069e+00 -1.538e+00 -2.099e+00 -2.024e+00 -6.355e-01 -2.011e+00 -3.512e+00 -1.297e+00
Grad in zero: -1.459e-01 1.532e-01 -2.660e+00 -2.843e+00 1.225e+00 4.960e-01 1.713e+00 -3.254e-01 4.139e-01 8.125e-01 2.091e+00 -1.719e+00 1.071e+00 -6.934e-01 7.520e-01
Step 1 1: loss=1.27452771e+03, test_loss=6.24037148e+04
Test derivatives: -8.238e-01 7.708e-01 5.783e-01 -1.634e+00 -2.157e-01 -5.364e-01 1.219e-01 -1.236e+00 -5.582e-01 -8.915e-01 -1.423e+00 -6.292e-02 -1.605e+00 -2.628e+00 -5.511e-01
Grad in zero: 5.782e-01 -1.297e+00 -4.494e+00 -5.736e+00 1.022e+00 2.724e-01 4.189e+00 -9.414e-01 -2.351e+00 3.674e+00 3.721e+00 -2.472e+00 2.096e+00 -3.843e-01 1.578e+00
Step 1 2: loss=1.22151111e+03, test_loss=6.16602852e+04
Test derivatives: 8.649e-01 3.307e+00 2.725e+00 1.649e-02 2.265e+00 1.182e+00 2.815e+00 4.359e-01 1.789e+00 1.321e+00 -1.873e-01

/tmp/ipykernel_48324/4014741422.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Step 1: loss=8.64578857e+02
  saved checkpoint: nn_data/from_scratch_tanh_adamw_128_1.pt
  saved training data: nn_data/from_scratch_tanh_adamw_128_1.npz
Step 2 0: loss=8.25725830e+02, test_loss=4.36126602e+04
Test derivatives: 4.930e+00 7.395e+00 3.771e+00 6.954e+00 2.832e+01 8.581e+00 1.822e+01 1.355e+01 1.723e+01 1.503e+01 8.492e+00 2.268e+00 3.515e+00 1.121e+01 8.914e+00
Grad in zero: 1.389e+01 9.248e+00 7.103e-01 -2.192e+01 1.723e+00 1.907e+01 -4.045e-01 -1.516e+00 5.551e+00 1.084e+01 -5.750e+00 -1.851e+01 1.341e+00 -8.884e-01 -1.010e+01
Step 2 1: loss=7.78329895e+02, test_loss=4.08120234e+04
Test derivatives: 5.759e+00 6.355e+00 2.860e+00 8.204e+00 3.086e+01 9.038e+00 1.861e+01 1.478e+01 1.744e+01 1.552e+01 8.194e+00 2.659e+00 3.588e+00 1.117e+01 9.779e+00
Grad in zero: 1.399e+01 1.044e+01 -4.247e-01 -2.111e+01 1.082e+00 2.512e+01 -9.574e-01 -2.475e+00 7.040e+00 1.216e+01 -1.038e+01 -2.012e+01 3.909e+00 6.647e-01 -1.304e+01
Step 2 2: loss=7.36552734e+02, test_loss=3.82988594e+04
